## FOV and Timepoint Normalization

### Imports

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import scipy as sp
import dask.dataframe as dd
import pandas as pd
from dask.distributed import wait
import shutil

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [32]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

### Functions for this notebook

In [33]:
def get_aligned_loc_from_index(df,idx_series):
    df_out = df.loc[idx_series.tolist()]
    return df_out

def index_loc_lookup(df,idx_series):
    df_out = dd.map_partitions(get_aligned_loc_from_index,df,idx_series,align_dataframes=False)
    return df_out

def hrm_find_mode(series, max_iter=1000, min_binsize=50):
    working_series = series
    for i in range(max_iter):
        range_max,range_min = np.max(working_series),np.min(working_series)
        midpoint = (range_max+range_min)/2
        above_middle = working_series[working_series>midpoint]
        below_middle = working_series[working_series<=midpoint]
        
        count_above = len(above_middle)
        count_below = len(below_middle)
        
        
        if count_above>count_below:
            working_series = above_middle
        else:
            working_series = below_middle
        
        if i > 0:
            if (len(working_series)<min_binsize) or (last_midpoint==midpoint):
                return np.mean(working_series)

        last_midpoint = midpoint
        
def bootstrap_hrm(series, n_bootstraps=100, max_n_per_bootstrap=100):
    modes = []
    
    series_len = len(series)
    
    n_per_bootstrap = min(series_len,max_n_per_bootstrap)

    for n in range(n_bootstraps):
        modes.append(hrm_find_mode(series.sample(n=n_per_bootstrap)))
    return np.mean(modes)

def get_normal_fovs(fov_series, med_filter_size=5, n_stds=2, max_filter_dev=0.05):
    median_series = sp.ndimage.median_filter(fov_series,size=(med_filter_size,),mode="mirror")
    
    residuals = (fov_series-median_series)/np.mean(median_series)
    gaussian_fit = sp.stats.norm.fit(residuals)
    gaussian_fit = sp.stats.norm(loc=gaussian_fit[0], scale=gaussian_fit[1])

    lower,upper = (-n_stds*gaussian_fit.std(),n_stds*gaussian_fit.std())
    lower,upper = min(-max_filter_dev,lower),max(max_filter_dev,upper)
        
    thr_mask = ((residuals>lower)&(residuals<upper))
    
    return thr_mask

#### Start Dask

In [1]:
dask_controller = tr.trcluster.dask_controller(
    walltime="1:00:00",
    local=False,
    n_workers=100,
    n_workers_min=100,
    memory="16GB",
    working_directory= dask_wd,
)
dask_controller.startdask()

NameError: name 'tr' is not defined

In [2]:
dask_controller.displaydashboard()

NameError: name 'dask_controller' is not defined

#### Subsample dataset (optional)

In [ ]:
# headpath = "/home/de64/scratch/de64/sync_folder/2022-06-11_lDE23_Run_2/Growth_Division"
# subsample_headpath = "/home/de64/scratch/de64/sync_folder/2022-06-11_lDE23_Run_2_TimeSubsample/Growth_Division"
# timecrop = tr.dataset_time_cropper(headpath,subsample_headpath,"fluorsegmentation")

In [ ]:
# timecrop.crop_dataset(dask_controller,list(range(0,133)),overwrite=True)

### Import Lineage dfs

In [ ]:
lineage_df_paths = ["/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/lineage/output/"]
lineage_df_list = [dd.read_parquet(path,calculate_divisions=True) for path in lineage_df_paths]

# #hack assuming both datasets are the same size, within a factor of 2
max_cellid = lineage_df_list[0]['CellID'].max().compute()
max_timepoint = lineage_df_list[0]['timepoints'].max().compute()
max_file_trench_idx = lineage_df_list[0]['File Trench Index'].max().compute()
max_file_idx = lineage_df_list[0]['File Index'].max().compute()

cellid_digits = len(str(max_cellid*2))
timepoint_digits = len(str(max_timepoint))
file_trench_idx_digits = len(str(max_file_trench_idx*2))
file_idx_digits = len(str(max_file_idx*2))
print(cellid_digits,timepoint_digits,file_trench_idx_digits,file_idx_digits)

for i in range(len(lineage_df_list)):
    lineage_df_list[i]["Experiment #"] = i
    lineage_df_list[i] = lineage_df_list[i].reset_index()
    
    parq_filemultiexp_idx = lineage_df_list[i].apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["File Index"]):0{file_idx_digits}n}{int(x["File Trench Index"]):0{file_trench_idx_digits}n}{int(x["timepoints"]):0{timepoint_digits}n}{int(x["CellID"]):0{cellid_digits}n}'),\
                                                     axis=1, meta=("Multi-Experiment File Parquet Index",int))
        
    lineage_df_list[i]["Multi-Experiment File Parquet Index"] = parq_filemultiexp_idx.persist()
    lineage_df_list[i] = lineage_df_list[i].set_index("Multi-Experiment File Parquet Index",sorted=True)

parq_filemultiexp_idx = lineage_df_list[i].apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["File Index"]):08n}{int(x["File Trench Index"]):04n}{int(x["timepoints"]):04n}{int(x["CellID"]):04n}'), axis=1)
    
lineage_df_merged = dd.concat(lineage_df_list)

lineage_df_merged["Multi-Experiment trenchid"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["trenchid"]):08n}'), axis=1, meta=("Multi-Experiment trenchid", int))
lineage_df_merged["Multi-Experiment timepoints"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["timepoints"]):08n}'), axis=1, meta=("Multi-Experiment timepoints", int))
lineage_df_merged["Multi-Experiment fov"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["fov"]):08n}'), axis=1, meta=("Multi-Experiment fov", int))

dd.to_parquet(lineage_df_merged, "/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage",engine='pyarrow',compression='gzip',write_metadata_file=True,overwrite=True)

In [ ]:
lineage_df_merged.columns

In [ ]:
# ##temp fix
# lineage_df["CellID"] = lineage_df["CellID"].astype(int)
# lineage_df["Global CellID"] = lineage_df["Global CellID"].astype(int)

### Scaling to uninduced values
 - Comparing measured distribution to the dataset-wide distribution to rescale (measuring both mean and variance)
     - actually got rid of the variance contribution to the scaling, since it is a less reliable estimator
     - especially falls apart when the lineage gets a little heterogenous
 - Could be useful for aligning datasets later
 - Also consider having the derivative values (volume, surface area) recalculated based on the adjustment

In [3]:
dask_controller.shutdown()

NameError: name 'dask_controller' is not defined

In [38]:
crispr_induction_timepoint = 12
sample_rate_for_plots = 0.005

# values_to_rescale = ['mCherry mean_intensity', 'area', 'Length', 'Width', "Volume", "Surface Area"]
values_to_rescale = ['mCherry mean_intensity', 'area', 'major_axis_length', 'minor_axis_length', "Volume", "Surface Area"]

values_names = ['Mean mCherry Intensity','Area', 'Length', 'Width',"Volume","Surface Area"]

lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage",calculate_divisions=True)

In [39]:
subsetted_lineage_df = lineage_df[["fov","timepoints","Cell Y (local)","Cell X (local)","trenchid","Multi-Experiment trenchid","Multi-Experiment timepoints","Multi-Experiment fov","Experiment #"] + values_to_rescale]
subsetted_lineage_df_early = subsetted_lineage_df[subsetted_lineage_df["timepoints"]<crispr_induction_timepoint]

subsetted_lineage_df_trenchid_sorted = subsetted_lineage_df.reset_index(drop=False).set_index("Multi-Experiment trenchid").persist()
subsetted_lineage_df_early_trenchid_sorted = subsetted_lineage_df_early.set_index("Multi-Experiment trenchid")

subsetted_lineage_df_early_subsampled = subsetted_lineage_df_early.sample(frac=sample_rate_for_plots).compute()
subsetted_lineage_df_early_subsampled_trenchid_sorted = subsetted_lineage_df_early_subsampled.set_index("Multi-Experiment trenchid").sort_index()

In [ ]:
lineage_df_merged.get_partition()['Volume']

In [ ]:
len(exp_subset['Volume'])

In [ ]:
exp_subset['Volume'].isna().sum()

In [ ]:
nfig = plt.figure(figsize=(30,30))

for i,label in enumerate(values_to_rescale):
    print(i,label)
    trench_means = subsetted_lineage_df_early_trenchid_sorted[label].groupby("Multi-Experiment trenchid").apply(lambda x: np.mean(x),meta=(label,float)).compute()
    global_median_of_means = np.median(trench_means)
    rescaling_lookup = global_median_of_means/trench_means
    rescaling_lookup = rescaling_lookup.to_frame()
    rescaling_lookup.columns = [label + " Rescaling"]
    
    subsetted_lineage_df_trenchid_sorted_label_dd = subsetted_lineage_df_trenchid_sorted[label].to_frame().join(rescaling_lookup)
    rescaled_subsetted_lineage_df_trenchid_sorted = subsetted_lineage_df_trenchid_sorted_label_dd[label]*subsetted_lineage_df_trenchid_sorted_label_dd[label + " Rescaling"]
    rescaled_subsetted_lineage_df_trenchid_sorted.name = label
    
    rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted_label_dd = subsetted_lineage_df_early_subsampled_trenchid_sorted[label].to_frame().join(rescaling_lookup)
    rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted = rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted_label_dd[label]*\
                                                                     rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted_label_dd[label + " Rescaling"]
    rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted.name = label
        
    subsetted_lineage_df_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_trenchid_sorted
    subsetted_lineage_df_trenchid_sorted[label + " Rescaling"] = subsetted_lineage_df_trenchid_sorted_label_dd[label + " Rescaling"]
    subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted
    subsetted_lineage_df_early_subsampled_trenchid_sorted[label + " Rescaling"] = rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted_label_dd[label + " Rescaling"]
    
    experiment_list = list(subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"].unique())
    
    plt.subplot(4,4,i+1)
    for exp_num in experiment_list:
        exp_subset = subsetted_lineage_df_early_subsampled_trenchid_sorted[subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"] == exp_num]
        plt.hist(exp_subset[label],
             range=(0,np.nanpercentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="Experiment " + str(exp_num), density=True)
        
    plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],
             range=(0,np.nanpercentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="All Experiments", density=True)
    plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label],
             range=(0,np.nanpercentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="All Experiments Rescaled", density=True)
    plt.ylabel("# Cells", fontsize=18)
    plt.xlabel(values_to_rescale[i], fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.legend()
                      
plt.savefig("./Trench_correction_replicate_1.png",dpi=500)
plt.show()

In [ ]:
nan_filter_trench_series = subsetted_lineage_df_trenchid_sorted.groupby("Multi-Experiment trenchid").apply(lambda x: np.any(np.isnan(x[["Rescaled: " + name for name in values_to_rescale]]))).compute()
nonan_trenchids = nan_filter_trench_series[~nan_filter_trench_series].index.tolist()
subsetted_lineage_df_trenchid_sorted_filtered = subsetted_lineage_df_trenchid_sorted.loc[nonan_trenchids]
dd.to_parquet(subsetted_lineage_df_trenchid_sorted_filtered,"/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage_temp",\
              engine='pyarrow',compression='gzip',write_metadata_file=True,overwrite=True)
dask_controller.reset_worker_memory()

### Variables over FOVs
- only eliminating fovs with outlier values

In [ ]:
lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage_temp",calculate_divisions=True)
lineage_df_subset = lineage_df.repartition(npartitions=300).drop(columns = values_to_rescale + ["Cell Y (local)","Cell X (local)"]\
                                                    ).rename(columns={"Rescaled: " + name:name for name in values_to_rescale}).persist()
wait(lineage_df_subset);
fig = plt.figure(figsize=(30,20))
normal_fov_list = []
lineage_df_subsample = lineage_df_subset[lineage_df_subset["timepoints"]<crispr_induction_timepoint]

for i,label in enumerate(values_to_rescale):

    plt.subplot(4,4,i+1)
    
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("FOV #", fontsize=18)
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.ylim(0.,2.)
    
    fov_series_groupby = lineage_df_subsample.groupby("Multi-Experiment fov",sort=False)[label]
    fov_median_series = fov_series_groupby.apply(lambda x: np.median(x), meta=float).compute().sort_index()

    normal_fov_series = get_normal_fovs(fov_median_series)
    fov_median_series = fov_median_series[normal_fov_series]
    normal_fovs = set(fov_median_series.index)
    normal_fov_list.append(normal_fovs)
    
    plt.plot((fov_median_series/np.median(fov_median_series)).values)
    ##maybe add a line to divide the experiments
    plt.legend()
filtered_fov_list = list(set.intersection(*normal_fov_list))

lineage_df_fov_correction = lineage_df_subset[["Multi-Experiment File Parquet Index","Multi-Experiment timepoints","Multi-Experiment fov"] + values_to_rescale + \
                            [label + " Rescaling" for label in values_to_rescale]]
lineage_df_fov_correction = lineage_df_fov_correction[lineage_df_fov_correction["Multi-Experiment fov"].isin(filtered_fov_list)].persist()

plt.savefig("./FOV_check_replicate_1.png",dpi=500)

### Variables over time

In [43]:
target_samples = 100000 
correct_time = True

values_to_rescale_step_2 = ["mCherry mean_intensity"]
normalization_fn = [np.max]
ttl_samples = len(lineage_df_fov_correction)
frac_to_sample = target_samples/ttl_samples
lineage_df_subsample = lineage_df_fov_correction.sample(frac=frac_to_sample).compute()

In [ ]:
fig = plt.figure(figsize=(30,20))
values_names = ['Mean mCherry Intensity']
for i,label in enumerate(values_to_rescale_step_2):
    time_series_groupby = lineage_df_subsample.groupby("Multi-Experiment timepoints")[label]
    time_mode_series = time_series_groupby.apply(lambda x: bootstrap_hrm(x)).sort_index()
    time_correction_series = (normalization_fn[i](time_mode_series))/time_mode_series
    time_correction_dict = time_correction_series.to_dict()
    plt.subplot(2,3,i+1)
    plt.plot(time_correction_series.values)
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("Timepoint (5 min steps)", fontsize=18)
    #line seperating experiments
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18) 
    plt.ylim(0.,2.)
    if correct_time:
        lineage_df_fov_correction[label + ": Time Corrected"] = lineage_df_fov_correction.apply(lambda x: x[label]*time_correction_dict[x["Multi-Experiment timepoints"]], meta=float, axis=1).persist()
        lineage_df_fov_correction[label + " Rescaling"] = lineage_df_fov_correction.apply(lambda x: x[label + " Rescaling"]*time_correction_dict[x["Multi-Experiment timepoints"]], meta=float, axis=1).persist()
plt.savefig("./Time_correction_replicate_1.png",dpi=500)

### Overwrite Variables with Correction

In [ ]:
"/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage"
lineage_df_paths = ["/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/lineage/output/"]

In [45]:
lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Merged_Lineage",calculate_divisions=True)

lineage_df_fov_correction = lineage_df_fov_correction.reset_index().set_index("Multi-Experiment File Parquet Index",sorted=True).persist()
wait(lineage_df_fov_correction);
lineage_df_fov_correction.index = lineage_df_fov_correction.index.astype(int)
lineage_df_fov_correction = lineage_df_fov_correction.repartition(divisions=lineage_df.divisions,force=True).persist()
wait(lineage_df_fov_correction);
corrected_lineage_df  = index_loc_lookup(lineage_df,lineage_df_fov_correction.index)

for label in values_to_rescale:
    corrected_lineage_df[label] = lineage_df_fov_correction[label]
    corrected_lineage_df[label + " Rescaling"] = lineage_df_fov_correction[label + " Rescaling"]
if correct_time:
    for label in values_to_rescale_step_2:
        corrected_lineage_df[label] = lineage_df_fov_correction[label + ': Time Corrected']

corrected_lineage_df = corrected_lineage_df[['fov',
                                             'row',
                                             'trench',
                                             'trenchid',
                                             'timepoints',
                                             'Mother',
                                             'File Index',
                                             'File Trench Index',
                                             'CellID',
                                             'Global CellID',
                                             'Trench Score',
                                             'Mother CellID',
                                             'Daughter CellID 1',
                                             'Daughter CellID 2',
                                             'Sister CellID',
                                             'Centroid X',
                                             'Centroid Y',
                                             'Cell Y (local)',
                                             'Cell X (local)',
                                             'Kymograph File Parquet Index',
                                             'Kymograph FOV Parquet Index',
                                             'FOV Parquet Index',
                                             'File Parquet Index',
                                             'Experiment #'] + values_to_rescale + [label + " Rescaling" for label in values_to_rescale]]

In [46]:
lineage_df_paths = ["/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/lineage/output/"]
lineage_df_list = [dd.read_parquet(path,calculate_divisions=True) for path in lineage_df_paths]
output_path_list = ["/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_FOV_and_Time_Normalization"]
for i,df in enumerate(lineage_df_list):
    df_mask = (corrected_lineage_df['Experiment #'] == i).persist()
    corrected_lineage_df_masked = corrected_lineage_df[df_mask]
    corrected_lineage_df_masked = corrected_lineage_df_masked.set_index('File Parquet Index',sorted=True)
    corrected_lineage_df_masked.divisions = corrected_lineage_df_masked.clear_divisions().compute_current_divisions()
    corrected_lineage_df_masked = corrected_lineage_df_masked.repartition(divisions=df.divisions,force=True)
    corrected_lineage_df_masked.to_parquet(output_path_list[i], engine="pyarrow", overwrite=True)

### Shut down Dask

In [ ]:
dask_controller.shutdown()